<p>
<font size='5' face='Georgia, Arial'>IIC2233 Apunte Programación Avanzada</font><br>
<font size='1'> Modificado en 2024-1 al 2026-2 por Equipo Docente IIC2233. </font>
</p>

# Tabla de contenidos

1. [Aplicación del lado del servidor](#servidor)
2. [WSGI](#wsgi)
3. [Flask](#flask)
    1. [Ejemplo básico](#flask-basico)
    2. [Endpoints y parámetros](#flask-avanzado)
    3. [Datos JSON y validación](#json-servidor)
    4. [Respuestas y errores](#errores-servidor)

<a id="servidor"></a>
## Aplicación del lado del servidor

El servidor recibe solicitudes, ejecuta lógica de aplicación y genera respuestas. Puede estar implementado en cualquier lenguaje mientras respete el protocolo acordado.

**WSGI** es la interfaz estándar entre una aplicación web Python y un servidor WSGI. Flask implementa la aplicación; servidores como Waitress o Gunicorn pueden ejecutarla en producción. Los servicios de nube pueden ofrecer infraestructura, plataformas o productos administrados, por lo que no siempre pertenecen a una única categoría.

<a id="wsgi"></a>
### Ejemplo sin Flask

El script `codigo/ejemplo_sin_flask/servidor-API.py` implementa una aplicación WSGI mínima.

1. Ejecútalo en una terminal.
2. Prueba las siguientes tres celdas.
3. Presiona `Ctrl+C` en la terminal para detenerlo **antes** de iniciar los ejemplos Flask, porque todos usan el puerto 4444.

El ejemplo valida métodos y rutas para responder `404` y `405` cuando corresponde.

In [1]:
import requests


BASE_URL = "http://localhost:4444/"
respuesta = requests.get(BASE_URL, timeout=5)
respuesta.raise_for_status()
respuesta.status_code

200

In [2]:
respuesta.json()

{'mensaje': 'Hello World'}

In [3]:
# ¡Vamos a despedirnos haciendo una consulta a otro endpoint de nuestra API!
respuesta = requests.get(BASE_URL + "goodbye/")
print(respuesta.status_code, respuesta.json())

200 {'mensaje': 'Que la fuerza esté contigo'}


## Flask

En esta sección crearemos una API con Flask. Para instalar Flask y Requests en el entorno del kernel utiliza `%pip`. En un proyecto reproducible también conviene declarar las dependencias en un archivo.

In [4]:
%pip install flask requests


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: /Users/tamavidal/Desktop/Progra Avanzada/2026-2/contenidos-2026-2/semana-10-serializacion_y_networkin_1/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


### Ejemplo básico

La sintaxis básica de Flask asocia una función con una ruta y uno o más métodos HTTP mediante `@app.route`.

`app.run` inicia el servidor de desarrollo de Werkzeug. Es apropiado para ejercicios locales, pero no para producción. Una aplicación pública debe ejecutarse con un servidor WSGI de producción o una plataforma compatible.

**Probando la API**

La siguiente celda inicia el servidor directamente desde el notebook (sin necesidad de abrir una terminal). Una vez iniciado, dispondremos los siguientes _endpoints_:
- (`GET`) `http://localhost:4444/`:  ejecutará la función `hello_world`.
- (`GET`) `http://localhost:4444/numero_aleatorio`:  retornará un número aleatorio entre **-4444 y -11**.
- (`POST`) `http://localhost:4444/numero_aleatorio`:  retornará un número aleatorio entre **0 y 6**.


In [5]:
import logging
import random
import threading

from flask import Flask
from flask import request as flask_req
from werkzeug.serving import make_server


logging.getLogger("werkzeug").setLevel(logging.ERROR)

app_1 = Flask("servidor_1")


@app_1.route("/", methods=["GET"])
def hello_world():
    return {"texto": "Hello, World!"}


@app_1.route("/numero_aleatorio", methods=["GET", "POST"])
def numero_aleatorio():
    if flask_req.method == "POST":
        numero = random.randint(0, 6)
        return {"texto": f"Tu número es: {numero}", "método": "POST"}

    numero = random.randint(-4444, -11)
    return {"texto": f"Tu número es: {numero}", "método": "GET"}


servidor_1 = make_server("localhost", 4444, app_1)
hilo_servidor_1 = threading.Thread(
    target=servidor_1.serve_forever,
    daemon=True,
)
hilo_servidor_1.start()
print("Servidor 1 iniciado en http://localhost:4444")

Servidor 1 iniciado en http://localhost:4444


In [6]:
import requests

respuesta = requests.get("http://localhost:4444/")
respuesta.json()

{'texto': 'Hello, World!'}

In [7]:
respuesta = requests.get("http://localhost:4444/numero_aleatorio")
respuesta.json()

{'método': 'GET', 'texto': 'Tu número es: -1047'}

In [8]:
respuesta = requests.post("http://localhost:4444/numero_aleatorio")
respuesta.json()

{'método': 'POST', 'texto': 'Tu número es: 1'}

In [9]:
servidor_1.shutdown()
servidor_1.server_close()
hilo_servidor_1.join(timeout=2)
print("Servidor 1 detenido")

Servidor 1 detenido


<a id="flask-avanzado"></a>
### Ejemplo avanzado

El ejemplo avanzado cubre:

- parámetros dinámicos en la ruta;
- parámetros del *query string*;
- recepción y validación de JSON;
- respuestas y códigos de error.

Cuando una entrada del cliente es inválida, el servidor debe responder con un error 4xx, no permitir que una excepción termine convertida en `500 Internal Server Error`.

In [10]:
import logging
import threading

from flask import Flask, Response, abort
from flask import request as flask_req
from werkzeug.serving import make_server


logging.getLogger("werkzeug").setLevel(logging.ERROR)
app_2 = Flask("servidor_2")


@app_2.route("/hi/<string:username>", methods=["GET"])
def saludar(username):
    return {"texto": f"Hola {username}!"}


@app_2.route("/argumentos", methods=["GET"])
def argumentos():
    nombre = flask_req.args.get("nombre", default="Guillermo", type=str)
    especie = flask_req.args.get("especie", default="Gato", type=str)
    edad = flask_req.args.get("edad", default=0, type=int)
    return {"texto": f"Hola {especie} {nombre} de {edad} años!"}


@app_2.route("/body", methods=["POST"])
def datos_en_body():
    body_data = flask_req.get_json()

    if not isinstance(body_data, dict):
        return {"error": "El cuerpo debe ser un objeto JSON."}, 400
    if not {"var_1", "var_2"} <= body_data.keys():
        return {"error": "Faltan var_1 o var_2."}, 400

    numero_1 = body_data["var_1"]
    numero_2 = body_data["var_2"]
    if not isinstance(numero_1, (int, float)) or not isinstance(
        numero_2,
        (int, float),
    ):
        return {"error": "Las variables deben ser números."}, 422

    return {
        "var_1": numero_1,
        "var_2": numero_2,
        "result": numero_1 + numero_2,
    }


@app_2.route("/error/<int:instruccion>", methods=["DELETE"])
def error(instruccion):
    if instruccion == 0:
        abort(400)
    if instruccion == 1:
        return Response(status=418)
    if instruccion == 2:
        return Response(
            status=200,
            response="Podemos mandar textos",
            content_type="text/plain; charset=utf-8",
        )
    if instruccion == 3:
        return {"mensaje": "También JSON"}
    abort(404)


servidor_2 = make_server("localhost", 4444, app_2)
hilo_servidor_2 = threading.Thread(
    target=servidor_2.serve_forever,
    daemon=True,
)
hilo_servidor_2.start()
print("Servidor 2 iniciado en http://localhost:4444")

Servidor 2 iniciado en http://localhost:4444


#### Caso 1: _endpoint_ dinámico

```python
@app.route("/hi/<string:username>", methods=["GET"])
def saludar(username):
    return {"texto": f"Hola {username}!"}
```

En este caso, cuando se hace `"<string:username>"` dentro del _endpoint_, se interpreta que dicha sección de la URL puede ir mutando. Por ejemplo, hacer `"/hi/Luna"`, `"/hi/Cachirulo"`, `"/hi/Guillermo"` harán _match_ con este _endpoint_ y ejecutarán la función `saludar`.

A continuación vamos a probar este _endpoint_ con diferentes nombres:


In [11]:
import requests

respuesta = requests.get("http://localhost:4444/hi/Luna")
respuesta.json()

{'texto': 'Hola Luna!'}

In [12]:
respuesta = requests.get("http://localhost:4444/hi/Cachirulo")
respuesta.json()

{'texto': 'Hola Cachirulo!'}

In [13]:
respuesta = requests.get("http://localhost:4444/hi/Guillermo")
respuesta.json()

{'texto': 'Hola Guillermo!'}

#### Caso 2: argumentos en la URL

`request.args` es un `ImmutableMultiDict` con los parámetros del *query string*. Su método `get` permite indicar valor por defecto y conversión de tipo.

Por ejemplo, `/argumentos?nombre=Lucky` entrega `nombre=Lucky`. Los parámetros no utilizados pueden ignorarse o rechazarse según el contrato de la API.

In [14]:
respuesta = requests.get("http://localhost:4444/argumentos?nombre=Pepa&especie=Tortuga&edad=70")
respuesta.json()

{'texto': 'Hola Tortuga Pepa de 70 años!'}

In [15]:
respuesta = requests.get("http://localhost:4444/argumentos?nombre=Cachirulo&edad=5")
respuesta.json()

{'texto': 'Hola Gato Cachirulo de 5 años!'}

In [16]:
respuesta = requests.get("http://localhost:4444/argumentos")
respuesta.json()

{'texto': 'Hola Gato Guillermo de 0 años!'}

In [17]:
parametros = {"nombre": "Luna", "edad": 8, "informacion_extra": "Le gusta el churu"}

respuesta = requests.get("http://localhost:4444/argumentos", params=parametros)
respuesta.json()

{'texto': 'Hola Gato Luna de 8 años!'}

<a id="json-servidor"></a>
#### Caso 3: datos JSON y validación

`request.get_json()` interpreta el cuerpo únicamente cuando su tipo de contenido indica JSON. El cliente debe enviar `Content-Type: application/json`; con Requests se obtiene usando `json=body`.

El endpoint valida estructura y tipos para responder `400` o `422` en vez de provocar un error interno.

In [18]:
body = {"var_1": 11, "var_2": 13}

respuesta = requests.post(
    "http://localhost:4444/body",
    json=body,
    timeout=5,
)
respuesta.raise_for_status()
respuesta.json()

{'result': 24, 'var_1': 11, 'var_2': 13}

In [19]:
body = {"var_1": 4444, "var_2": 0}

respuesta = requests.post(
    "http://localhost:4444/body",
    json=body,
    timeout=5,
)
respuesta.raise_for_status()
respuesta.json()

{'result': 4444, 'var_1': 4444, 'var_2': 0}

<a id="errores-servidor"></a>
#### Caso 4: respuestas y errores

`abort(codigo)` detiene la vista mediante una `HTTPException`. `Response` permite controlar cuerpo, código y `Content-Type`. Para JSON, Flask puede convertir directamente un diccionario.

El código `202 Accepted` solo corresponde cuando el procesamiento todavía no termina; por eso la respuesta JSON inmediata utiliza `200 OK`. Las instrucciones desconocidas responden `404`.

In [20]:
respuesta = requests.delete("http://localhost:4444/error/0")
respuesta

<Response [400]>

In [21]:
respuesta = requests.delete("http://localhost:4444/error/1")
respuesta

<Response [418]>

In [22]:
respuesta = requests.delete("http://localhost:4444/error/2")
print(respuesta)
print(respuesta.text)

<Response [200]>
Podemos mandar textos


In [23]:
respuesta = requests.delete("http://localhost:4444/error/3")
print(respuesta)
print(respuesta.json())

<Response [200]>
{'mensaje': 'También JSON'}


In [24]:
servidor_2.shutdown()
servidor_2.server_close()
hilo_servidor_2.join(timeout=2)
print("Servidor 2 detenido")

Servidor 2 detenido
